# 01 · Fine-tune ProtBERT-BFD into AMP-BERT

Refactor of the original `fine-tune_with_amps.ipynb`. Fine-tunes the pre-trained
**ProtBERT-BFD** language model on the Veltri training set
(`data/raw/all_veltri.csv`); the resulting fine-tuned model **is** AMP-BERT and is
saved to `models/amp_bert/`.

All reusable logic lives in `src/amp_bert/`.

In [ ]:
# --- bootstrap: works locally and on Google Colab ---
import sys, pathlib

REPO = "https://github.com/BioGavin/AMP-BERT.git"
try:
    import amp_bert  # already installed / on sys.path
except ModuleNotFoundError:
    root = pathlib.Path.cwd()
    root = root.parent if root.name == 'notebooks' else root
    src = root / 'src'
    if not (src / 'amp_bert').exists():
        # fresh Colab runtime: clone the repo
        import subprocess
        subprocess.run(['git', 'clone', '--depth', '1', REPO], check=True)
        src = pathlib.Path('AMP-BERT') / 'src'
    sys.path.insert(0, str(src))

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
from transformers import Trainer
from amp_bert import (AmpDataset, load_dataset, compute_metrics,
                      model_init, build_training_args)
from amp_bert.config import TRAIN_CSV, MODELS_DIR, RESULTS_DIR

## Load training data

In [ ]:
df = load_dataset(TRAIN_CSV, shuffle=True, random_state=0)
print(df.shape)
df.head(7)

In [ ]:
train_dataset = AmpDataset(df, max_len=200)
len(train_dataset)

## Train

Effective batch size = 1 × 64 (gradient accumulation) = 64, 15 epochs — as in the paper.
On a single GPU this takes several hours; reduce `num_train_epochs` for a smoke test.

In [ ]:
out_dir = str(RESULTS_DIR / 'amp_bert_train')
training_args = build_training_args(out_dir, num_train_epochs=15, seed=0)

trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=train_dataset,
    compute_metrics=compute_metrics,
)
trainer.train()

## Sanity check on the training data, then save

In [ ]:
_, _, metrics = trainer.predict(train_dataset)
print(metrics)

In [ ]:
save_path = MODELS_DIR / 'amp_bert'
trainer.save_model(str(save_path))
print('saved to', save_path)